LLM CON API (OPEN AI, HUGGING FACE HUB)

Finora, con model=AutoModelForCasualLM.from_pretrained(...), il modello veniva caricato nella RAM/VRAM del computer
Con API il tuo programma, via HTTPS utilizza LMM tramite il server del provider, attende la risposta per essere processata dal programma.
Quindi, non gestisci GPU, pesi del modello, caricamento in memoria e inferenza.
Gestisci: input, autonticazione, parametri, risposta, errori, timeout, costi.

API = Applicazione Programming Interface

Il programma dice al servizio:
"Questa è la mia richiesta. Elaborala e dammi il risultato"
Esempio:
JSON
{
    "model": "...",
    "input": "Spiegami cos'è un tensore"
}
e il server restituisce:
JSON
{
    "output": "Un tensore è..."
}
Sotto Python, normalmente, c'è una richiesta HTTP

OPENAI API
Oggi l'API OpenAI utilizza come interfaccia principale la Responses API per molte applicazioni moderne. L'SDK Python permette di creare un client e inviare una richiesta con client.responses.create(...)

from openai import OpenAI
client = OpenAI()
response = client.responses.create(
    model="MODELLO",
    input="Spiegami cos'è un tensore."
)
print(response.output_text)

Ma l'API come sa chi sono?
Serve una API Key
Ma attenzione, non scrivere mai la key nel codice, meglio una variabile ambiente (es. OPENAI_API_KEY)
Ricorda di escludere il materiale con le chiavi, tramite .gitignore

Perchè utilizzare un API invece di caricare il modello?
Con un modello locale: devi avere hardware, devi avere RAM/VRAM, devi scaricare i pesi, devi gestire runtime, devi aggiornare il modello
Con API Python utilizzi il modello remoto.
Tu consumi il servizio, ed in cambio accetti: costo per utilizzo, latenza di rete, dipendenza dal provider, necessità di connessione, considerazioni di privacy
Quindi API non è sempre meglio del locale, sono architetture differenti da valutare caso per caso.

OpenAI supporta Structured Output, cioè la possibilità di vincolare l'output a uno schema JSON definito dall'applicazione. E' da distinguere questo caso dal semplice JSON mode.
Structured Output cerca di garantire l'aderenza allo schema, non solo che il JSON sia sintatticamente valido.

Puoi anche usare Pydantic
e chiedere al modello di produrre qualcosa compatibile con Order

from pydantic import BaseModel
class OrderLine(BaseModel):
    item: str
    quantity: int
class Order(BaseModel):
    customer: str | None
    lines: list[OrderLine]

API + function calling


Non vuoi solo LLM -> risposta.
Ma: LLM -> 'devo cercare il cliente' -> chiamata get_customer()

Puoi descrivere al modello una funzione es. get_customer_price() con detarminati parametri, es. customer_code, item_code, quantity
Il modello potrebbe richiedere:
{
    "customer_code": "STAUFF",
    "item_code": "ABC123",
    "quantity": 500
}
Il modello non esegue direttamente la funzione SQL
- LLM richiede la funzione
- Python riceve la richiesta
- Python esegue SQL
- Python restituisce il risultato al modello
- LLM continua

Il modello può produrre una o più funcion call, la tua applicazione la esegue e rimanda i risultati al modello.

Hugging-Face-Hub

Il Hub è prima di tutto un enorme repository: di modelli, di dataset, di tokenizer, di adapter, di spaces.
Esempio 
Qwen/...
meta-llama/...
google/...

Il fatto che un modello esista sul Hub, non significa auotomaticamente, che sia disponibile come servizio API remoto pronto all'uso.
Per l'inference Hugging Face oggi distingue tra: inference Providers, inference Endpoints, inference Locale.
huggingsface_hub.InferenceClient, fornisce un interfaccia comune per questi scenari, e può interagire sia con provider remoti sia con endpoint dedicati e perfino server locali come Ollama, llama, vLLM, TGI.

Hugging-Face Inference Providers
L'architettura è grossomodo: tuo programma -> hugging face -> inference provider -> modello.
Hugging-Face può funzionare da livello unificato davanti a diversi provider e centralizzare autenticazione e billing. Con selezione automatica può anche scegliere il provider disponibile per il modello richiesto.

from huggingface_hub import InferenceClient
import os
client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
    provider="auto")
response = client.chat.completions.create(
    model="MODELLO_HF",
    messages=[
        {
            "role": "user",
            "content": "Spiegami cos'è un tensore."
        }
    ]
)
print(response.choices[0].message.content)

Hugging-Face usa anche una sintassi compatibile con quella dei client OpenAI per le chat 

Gestione delle Chiavi e Sicurezza
Proteggere il perimetro operativo
L'integrazione di LLM tramite API richiede l'uso di chiavi crittografate che fungono da identità e metodo di pagamento. La loro compromissione può causare danni economici e fughe di dati. Una compromissione, non significa solo che qualcuno utilizzerà la vostra carta di credito, ma che qualcuno acceda a dati sensibili consultando lo storico delle chat.
Le Best practices per evitare l'hard-coding delle credenziali, utilizzano variabili d'ambiente e file di configurazione protetti.

Standard di Sicurezza
La sicurezza si basa su 3 pilastri fondamentali:
1) File .env: utilizzo di file locali non tracciati dai sistemi di versionamento per memorizzare le API Key
2) Libreria python-dotenv: caricamento dinamico, delle variabili d'ambiente nello spazio di memoria del processo Python, carica le chiavi solo nella memoria volatile
3) Gitignore: configurazione del file di esclusione per impedire l'output accidentale di segreti su repository pubblici. Impedire al file .env di finire sul github
La sicurezza dell'integrazione e inversamente proporzionale alla visibilità della chiave del codice sorgente.

Ambiente di Esecuzione
Quando passata dalla piccola start-up alla grande azienda, i file .env non bastano più, entrano in gioco sistemi che criptano i segreti e li distribuiscono in modo sicuro al server. La rotazione delle chiavi  è importanrte (rotazione delle chiavi)
Creare le nuovi chiavi con permessi granulari,  per ridurre il raggio d'azione di un leak (scope limiting).

Il rischio di Leak è reale e costante, per questo esistono scanner automatici che monitorano i commit per individuare pattern riconducibili a chiavi OpenAI o Hugging-Face.

Una volta blindata la sicurezza, dobbiamo decidere a quale motore affidarci.

Modelli Open vs Closed
Analisi strategica del provider
La scelta tra modelli proprietari (OpenAI, Anthropic) e modelli open-weight (Llama 3, Mistral) non è solo tecnica ma anche economica e normativa.
Importante è valutare non solo la tecnica ma anche economica e normativa.

Scegliere tra modelli open (Llama 3) e modelli close (GPT4), è una scelta difficile, da un lata la comodotà dell'ai più sicura
Vedremo come valutare questo trade off

Oltre al denaro c'è un valore che non ha prezzo
Privacy e Compliance
I modelli Open permettono di processare dati sensibili senza che questi lascino mai il perimetro aziendale, facilitando la conformità GDPR
Sui modelli Open è possibile eseguire fine-tuning granulari e ottimizzazioni hardware (quantizzazione) non accessibili tramite API chiuse.
Utilizzare modelli standard (come Llama) riduce la dipendenza da un singolo fornitore di servizi cloud. Se domani un provider decidere di triplicare i prezzi, è possibile trasferire il modello su un altro server, cambiando poco nel codice.

Analisi delle Performance Relative.
Benchmark vs Reale Utilizzo
Mentre i modelli chiusi dominano spesso i benchmark generali, un modello open specializzato può superarne l'efficienza su task verticali.
L'efficienza di un'integrazione si misura nel rapporto tra qualità della risposta e costo operativo per query.
A volte un modello piccolo e specializzato batte un modello gigante.

Se vuoi la potenza dei modelli open senza gestire l hardware c'è una terza soluzione

Serveless Inference
Sfruttare l'Hub di Hugging Face
La gestione di GPU fisiche è complessa e costosa.
Hugging Face offre una via di mezzo tra l'API chiusa e il self-hosting: l'interfaccia API.
Questa architettura serveless, permette di interrogare migliaia di modelli open senza preoccuparsi del provisioning dell'hardware sottostante.
Hugging-face si occupa di accendere i server necessari nel momento in cui arriva una richiesta.

Workflow di Integrazione
Uso della libreria 'huggingface_hub' per gestire programmaticamente il download e l'interrogazione dei modelli (hub python library).
Gestione degli errori 429 tramite strategie di backoff esponenziale per garantire la resilienza dell'applicazione (aspettando un tempo crescendi prima di riprovare) (rate limiting).
Facilità nel testare diversi modelli cambiando semplicemente l'ID dell'endpoint accelerando i cicli di A/B testing (model switching).

Scaling Efficace
Il bello del serveless elimina lo spreco, paghi solo i millisencondi in cui la GPU lavora per noi, non ci sono costi per la GPU inattiva, paghi solo per l'effettivo tempo di elaborazione.
Questa architettura permette di scalare da zero a migliaia di richieste senza interventi manuali sulla configurazione hardware.
Se l'app diventa virale passando da 10 a 10.000 utenti al minuto, l'infrastruttura server scala automaticamente in base al traffico (architettura elastica). 

Hugging-Face
hugging-Face può essere usato sia in locale sia in cloud.
Hugging Face Hub = posto da cui scarichi i modelli/dataset
Hugging Face Inference = servizio remoto che esegue il modello per te
se fai:
model = AutoModelForCausalLM.from_pretrained("Qwen/...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/...")
la prima volta i pesi vengono scaricati dall'Hub e messi nella cache locale. Dopo il download, il modello può girare interamente sul tuo pc/server: prompt, email, pdf e output non devono essere inviati a Hugging-Face. Transformers supporta esplicitamente modalità offline con HF_HUB_OFFLINE=1 oppure locale_file_only=True
Internet - scarico Qwen una volta da Hugging Face
Server aziendale - Qwen locale - email/ordini/prezzi/pdf
da questo momento l'inferenza resta locale.

Diverso è se utilizzi qualcosa come Hugging Face Inference Providers o un Inference Endpoint
In quel caso, il contenuto della richiesta viene inviato fuori dalla rete locale. Hugging Face descrive Inference Providers come un proxty che instrada le richieste verso provider esterni.

Le librerie di Huggin Face possono inviare alcune informazioni di telelmetria d'uso. Non significa che inviano automaticamente i tuoi prompt, ma se vuoi una configurazione aziendale realmente chiusa puoi disabilitare anche questo con:
HF_HUB_DISABLE_TELEMETRY=1
HF_HUB_OFFLINE=1

In [ ]:
# Verifica token
import os
from dotenv import load_dotenv

env_path = os.path.join(os.getcwd(), ".env")

load_dotenv(
    dotenv_path=env_path,
    override=True
)

api_token = os.getenv("HF_TOKEN", "").strip()

print("File .env:", env_path)
print("File esistente:", os.path.exists(env_path))
print("Token trovato:", bool(api_token))
print("Prefisso:", api_token[:3])
print("Lunghezza:", len(api_token))
print("È un token HF:", api_token.startswith("hf_"))

File .env: c:\EPICODE\Epicode_Python_AI_MachineLearning\PYTHON\Modulo_6_Generative\4_IntregrazioneLLM\.env
File esistente: True
Token trovato: True
Prefisso: hf_
Lunghezza: 37
È un token HF: True


In [15]:
# Verifica token

from huggingface_hub import whoami

try:
    account = whoami(token=api_token)

    print("Token valido.")
    print("Account Hugging Face:", account["name"])

except Exception as e:
    print("Token NON riconosciuto:")
    print(e)

Token valido.
Account Hugging Face: BarbaraUberti


In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

# Determiniamo il percorso assoluto della cartella dello script per caricare il file .env correttamente,
# indipendentemente da dove viene lanciato il comando nel terminale.
#script_dir = os.path.dirname(os.path.abspath(__file__))
# Nei notebook __file__ non esiste.
# Usiamo la cartella di lavoro corrente.
script_dir = os.getcwd()

env_path = os.path.join(script_dir, ".env")
load_dotenv(dotenv_path=env_path)

api_token = os.getenv("HF_TOKEN")

# Controllo preventivo dell'autenticazione.
# Verifichiamo immediatamente se il token e' stato caricato correttamente. Se manca,
# solleviamo un'eccezione per bloccare l'esecuzione prima di tentare chiamate API destinate al fallimento.
if not api_token:
    raise ValueError("Errore critico: la variabile HF_TOKEN non e' stata configurata nel file .env.")

# Configurazione del client Hugging Face.
# Proviamo con Qwen 2.5 7B Instruct, un modello molto recente e performante,
# solitamente ben supportato dall'API gratuita.
model_id = "Qwen/Qwen2.5-7B-Instruct" #questo modello va in errore, cambio
model_id = "deepseek-ai/DeepSeek-V3-0324"
client = InferenceClient(model=model_id, provider="auto", api_key=api_token)


def genera_social_post(testo_lungo):
    """
    Trasforma un testo complesso in un riassunto per i social media usando l'API Conversational.
    
    Vantaggi:
    - Separazione tra istruzioni di sistema (ruolo 'system') e input utente (ruolo 'user').
    - Maggiore coerenza nelle risposte del modello.
    """

    # Definiamo la struttura della conversazione.
    messages = [
        {"role": "system", "content": "Sei un esperto di social media. Riassumi i testi in post per X (max 20 parole)."},
        {"role": "user", "content": testo_lungo}
    ]
    
    try:
        # Chiamata alla Chat Completion API.
        # Restituisce un oggetto strutturato da cui estraiamo il contenuto del messaggio.
        response = client.chat_completion(
            messages=messages,
            max_tokens=60,
            temperature=0.7
        )
        
        # Estraiamo il testo della risposta.
        return response.choices[0].message.content
    except Exception as e:
        return f"Impossibile completare la richiesta. Dettaglio errore: {e}"

# Flusso principale di esecuzione del programma.
if __name__ == "__main__":
    # 1. Definizione dell'input: un breve testo descrittivo.
    articolo = "L'intelligenza artificiale nel 2026 ha raggiunto vette incredibili grazie alla scalabilità delle GPU."
    
    # 2. Feedback visivo per l'utente sulla console.
    print("Contattando il modello per la generazione del post...")
    
    # 3. Esecuzione della logica: passiamo i dati alla funzione dedicata.
    # Qui avviene l'interazione tra la logica di business dello script e il servizio API esterno.
    risultato = genera_social_post(articolo)
    
    # 4. Output finale: stampa del risultato ottenuto o dell'eventuale messaggio di errore.
    print("-" * 30)
    print(f"Contenuto generato:\n{risultato}")
    print("-" * 30)

Contattando il modello per la generazione del post...
------------------------------
Contenuto generato:
AI nel 2026: GPU scalabili portano a traguardi incredibili. 🚀 #Tech #Futuro
------------------------------
